# StreamFlix Content Analytics
## Phase 1 — Data Cleaning & Quality Check

**Project:** StreamFlix Content Analytics  
**Phase:** 1 — Data Cleaning & Quality Check  
**Analyst:** Kashif Siddique  
**Tools:** Python, Pandas, NumPy, Jupyter Notebook  

### Objective

To load, profile, validate, clean, and assess the quality and integrity of the six StreamFlix datasets before performing exploratory analysis and business KPI calculations.

The Phase 1 workflow follows the StreamFlix Project PRD, Data Dictionary, ERD, and SQL schema. The raw source files are treated as **read-only**.

> **Working principle:** All cleaning is performed on DataFrame copies. Records are only corrected or removed when a clear business rule or reliable reference supports the change. Ambiguous issues are flagged and documented rather than silently altered.

## Table of Contents

1. Environment Setup  
2. Import Libraries  
3. Load Datasets  
4. Initial Data Profiling  
   - 4.1 Dataset Shape  
   - 4.2 Current Data Types  
   - 4.3 Initial Profiling Observations  
5. Missing Value Analysis  
   - 5.1 Missing Counts and Percentages  
   - 5.2 Missing Value Interpretation  
6. Duplicate Analysis  
   - 6.1 Exact Duplicate Rows  
   - 6.2 Primary-Key Uniqueness Check  
7. Data Type Validation  
   - 7.1 Expected Analytical Types  
   - 7.2 Date Parsing Validation  
   - 7.3 Numeric Validation  
   - 7.4 Data Type Validation Summary  
8. Business & Data Quality Checks  
   - 8.1 Subscriber Churn-Date Rules  
   - 8.2 Short-Tenure Active Subscribers  
   - 8.3 Watch-Duration Outliers  
   - 8.4 Completion Percentage Validation  
   - 8.5 Review Sentiment Validation  
   - 8.6 Watch History Referential Integrity  
   - 8.7 Watchlist Referential Integrity  
   - 8.8 Ratings Validation  
   - 8.9 Catalogue Consistency Checks  
   - 8.10 Additional Temporal Consistency Checks  
9. Data Cleaning  
10. Post-Cleaning Validation  
11. Export Cleaned Data  
12. Data Quality Summary  
13. Phase 1 Conclusion  
14. Phase 1 Data Quality Report  
15. PRD Requirement Checklist  
16. Phase 1 Completion Status

## 1. Environment Setup

In [1]:
# ------------------------------------------------------------
# ENVIRONMENT SETUP
# Locate the raw StreamFlix CSV files automatically.
# This lets the notebook work both in our project folder
# and inside the ChatGPT workspace without changing paths manually.
# ------------------------------------------------------------

from pathlib import Path

EXPECTED_FILES = {
    "subscribers.csv",
    "titles.csv",
    "watch_history.csv",
    "ratings.csv",
    "reviews.csv",
    "watchlist.csv",
}

candidate_raw_dirs = [
    Path("../data/raw"),
    Path("data/raw"),
    Path("/mnt/data"),
]

RAW_DIR = None
for candidate in candidate_raw_dirs:
    if all((candidate / filename).exists() for filename in EXPECTED_FILES):
        RAW_DIR = candidate
        break

if RAW_DIR is None:
    raise FileNotFoundError(
        "Could not locate all 6 StreamFlix CSV files. "
        "Place them in ../data/raw/ or update RAW_DIR."
    )

if RAW_DIR.name == "raw" and RAW_DIR.parent.name == "data":
    PROJECT_ROOT = RAW_DIR.parent.parent
    CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"
    OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables" / "phase1"
else:
    PROJECT_ROOT = Path("/mnt/data/streamflix_phase1_workspace")
    CLEAN_DIR = PROJECT_ROOT / "cleaned"
    OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables" / "phase1"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data directory :", RAW_DIR.resolve())
print("Cleaned data output:", CLEAN_DIR.resolve())
print("QC output directory :", OUTPUT_DIR.resolve())

Raw data directory : C:\Users\DELL4\OneDrive\Documents\Internmo\StreamFlix Content Analytics Project\Data\Raw
Cleaned data output: C:\Users\DELL4\OneDrive\Documents\Internmo\StreamFlix Content Analytics Project\Data\Cleaned
QC output directory : C:\Users\DELL4\OneDrive\Documents\Internmo\StreamFlix Content Analytics Project\Outputs\Tables\phase1


## 2. Import Libraries

In [2]:
# ------------------------------------------------------------
# IMPORT LIBRARIES
# pandas  -> loading, cleaning, grouping, and validating tabular data
# numpy   -> numerical utilities
# display -> cleaner notebook output for tables and markdown summaries
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

REFERENCE_DATE = pd.Timestamp("2026-06-01")

print("Libraries imported successfully.")

Libraries imported successfully.


## 3. Load Datasets

The StreamFlix project contains six interconnected CSV files. `watch_history` is the central viewing fact table, while `subscribers` and `titles` are the main dimensions. `ratings`, `reviews`, and `watchlist` link subscribers to titles.

In [3]:
# ------------------------------------------------------------
# LOAD ALL SIX DATASETS
# Store each CSV inside a dictionary called `dfs`.
# This makes it easy to loop through every table using the same code
# instead of writing the same profiling logic six separate times.
# ------------------------------------------------------------

file_map = {
    "subscribers": RAW_DIR / "subscribers.csv",
    "titles": RAW_DIR / "titles.csv",
    "watch_history": RAW_DIR / "watch_history.csv",
    "ratings": RAW_DIR / "ratings.csv",
    "reviews": RAW_DIR / "reviews.csv",
    "watchlist": RAW_DIR / "watchlist.csv",
}

dfs = {
    name: pd.read_csv(path, low_memory=False)
    for name, path in file_map.items()
}

total_rows = sum(len(df) for df in dfs.values())

print(f"Datasets loaded : {len(dfs)}")
print(f"Total raw rows  : {total_rows:,}")

for name, df in dfs.items():
    print(f"{name:15s}: {len(df):,} rows × {len(df.columns)} columns")

Datasets loaded : 6
Total raw rows  : 979,000
subscribers    : 15,000 rows × 14 columns
titles         : 9,000 rows × 21 columns
watch_history  : 650,000 rows × 10 columns
ratings        : 130,000 rows × 5 columns
reviews        : 110,000 rows × 7 columns
watchlist      : 65,000 rows × 5 columns


### Dataset Loading Result

All six StreamFlix CSV files are loaded into separate DataFrames. No raw file is modified in this notebook. Any standardization performed later is applied to independent copies.

## 4. Initial Data Profiling

### 4.1 Dataset Shape

In [4]:
# ------------------------------------------------------------
# BASIC DATASET PROFILE
# Define the expected primary key for each table, then calculate:
# rows, columns, missing cells, duplicate rows, and duplicate PK values.
# This gives us a quick quality snapshot before any cleaning.
# ------------------------------------------------------------

PRIMARY_KEYS = {
    "subscribers": "subscriber_id",
    "titles": "title_id",
    "watch_history": "watch_id",
    "ratings": "rating_id",
    "reviews": "review_id",
    "watchlist": "watchlist_id",
}

profile_rows = []

for table, df in dfs.items():
    pk = PRIMARY_KEYS[table]
    profile_rows.append({
        "table": table,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "duplicate_primary_key_rows": int(df.duplicated(subset=[pk]).sum()),
    })

dataset_profile = pd.DataFrame(profile_rows)
display(dataset_profile)

dataset_profile.to_csv(OUTPUT_DIR / "dataset_profile.csv", index=False)

,table,rows,columns,missing_cells,exact_duplicate_rows,duplicate_primary_key_rows
0,subscribers,15000,14,11199,0,0
1,titles,9000,21,1966,0,0
2,watch_history,650000,10,0,0,0
3,ratings,130000,5,0,0,0
4,reviews,110000,7,0,0,0
5,watchlist,65000,5,0,0,0


### 4.2 Current Data Types

In [5]:
# ------------------------------------------------------------
# CURRENT DATA TYPES
# Inspect every column in every table and record:
# pandas dtype, non-null row count, and null row count.
# This helps us identify dates loaded as text and other type issues.
# ------------------------------------------------------------

dtype_rows = []

for table, df in dfs.items():
    for column in df.columns:
        dtype_rows.append({
            "table": table,
            "column": column,
            "pandas_dtype": str(df[column].dtype),
            "non_null_rows": int(df[column].notna().sum()),
            "null_rows": int(df[column].isna().sum()),
        })

current_dtypes = pd.DataFrame(dtype_rows)
display(current_dtypes)

current_dtypes.to_csv(OUTPUT_DIR / "current_data_types.csv", index=False)

,table,column,pandas_dtype,non_null_rows,null_rows
0,subscribers,subscriber_id,str,15000,0
1,subscribers,signup_date,str,15000,0
2,subscribers,country,str,15000,0
3,subscribers,region,str,15000,0
4,subscribers,age,int64,15000,0
5,subscribers,gender,str,15000,0
6,subscribers,plan_type,str,15000,0
7,subscribers,monthly_price_usd,float64,15000,0
8,subscribers,household_size,int64,15000,0
9,subscribers,primary_device,str,15000,0


### 4.3 Initial Profiling Observations

The profile establishes the baseline before cleaning. The following sections determine whether missing values are expected, whether keys are unique, whether dates and numeric fields parse correctly, and whether the relationships obey the supplied ERD.

## 5. Missing Value Analysis

### 5.1 Missing Counts and Percentages

In [6]:
# ------------------------------------------------------------
# MISSING VALUE ANALYSIS
# For every column, calculate both the number of null values
# and the percentage of rows affected.
# Percentages make it easier to compare tables of different sizes.
# ------------------------------------------------------------

missing_rows = []

for table, df in dfs.items():
    for column in df.columns:
        null_count = int(df[column].isna().sum())
        missing_rows.append({
            "table": table,
            "column": column,
            "null_count": null_count,
            "null_pct": round(null_count / len(df) * 100, 4),
        })

missing_summary = (
    pd.DataFrame(missing_rows)
    .sort_values(["null_count", "table", "column"], ascending=[False, True, True])
    .reset_index(drop=True)
)

display(missing_summary)
missing_summary.to_csv(OUTPUT_DIR / "missing_values.csv", index=False)

,table,column,null_count,null_pct
0,subscribers,churn_date,11199,74.6600
1,titles,license_expiry,1966,21.8444
2,ratings,rating,0,0.0000
3,ratings,rating_date,0,0.0000
4,ratings,rating_id,0,0.0000
5,ratings,subscriber_id,0,0.0000
6,ratings,title_id,0,0.0000
7,reviews,helpful_votes,0,0.0000
8,reviews,review_date,0,0.0000
9,reviews,review_id,0,0.0000


In [7]:
# Show only columns that actually contain missing values.
# This keeps the interpretation focused on fields needing attention.

columns_with_nulls = missing_summary[missing_summary["null_count"] > 0]
display(columns_with_nulls)

,table,column,null_count,null_pct
0,subscribers,churn_date,11199,74.6600
1,titles,license_expiry,1966,21.8444


### 5.2 Missing Value Interpretation

Two fields are expected to contain blanks by design:

- `subscribers.churn_date` should be blank for active subscribers.
- `titles.license_expiry` should be blank for StreamFlix Originals.

These nulls are therefore **not automatically treated as data-quality errors**. The later business-rule checks confirm whether they align with the documented logic.

## 6. Duplicate Analysis

### 6.1 Exact Duplicate Rows

In [8]:
# ------------------------------------------------------------
# EXACT DUPLICATE ROW CHECK
# df.duplicated() checks whether an entire row is repeated exactly.
# A value of 0 means no complete duplicate records were found.
# ------------------------------------------------------------

exact_duplicate_summary = pd.DataFrame([
    {
        "table": table,
        "exact_duplicate_rows": int(df.duplicated().sum())
    }
    for table, df in dfs.items()
])

display(exact_duplicate_summary)

,table,exact_duplicate_rows
0,subscribers,0
1,titles,0
2,watch_history,0
3,ratings,0
4,reviews,0
5,watchlist,0


### 6.2 Primary-Key Uniqueness Check

In [9]:
# ------------------------------------------------------------
# PRIMARY-KEY UNIQUENESS CHECK
# A primary key should uniquely identify each record.
# We therefore check duplicated values only within each table's PK column.
# ------------------------------------------------------------

pk_duplicate_rows = []

for table, df in dfs.items():
    pk = PRIMARY_KEYS[table]
    duplicate_count = int(df.duplicated(subset=[pk]).sum())
    pk_duplicate_rows.append({
        "table": table,
        "primary_key": pk,
        "duplicate_pk_rows": duplicate_count,
        "pk_unique": duplicate_count == 0,
    })

pk_duplicate_summary = pd.DataFrame(pk_duplicate_rows)
display(pk_duplicate_summary)

duplicate_summary = exact_duplicate_summary.merge(pk_duplicate_summary, on="table")
duplicate_summary.to_csv(OUTPUT_DIR / "duplicate_summary.csv", index=False)

,table,primary_key,duplicate_pk_rows,pk_unique
0,subscribers,subscriber_id,0,True
1,titles,title_id,0,True
2,watch_history,watch_id,0,True
3,ratings,rating_id,0,True
4,reviews,review_id,0,True
5,watchlist,watchlist_id,0,True


### Duplicate Analysis Conclusion

Primary keys are validated separately from exact-row duplicates because a duplicated key can represent a serious integrity issue even when the rest of the row differs.

## 7. Data Type Validation

### 7.1 Expected Analytical Types

In [10]:
# ------------------------------------------------------------
# EXPECTED ANALYTICAL DATA TYPES
# These dictionaries tell the notebook which columns should behave
# as dates and which should behave as numeric values during analysis.
# ------------------------------------------------------------

DATE_COLUMNS = {
    "subscribers": ["signup_date", "churn_date"],
    "titles": ["date_added", "license_expiry"],
    "watch_history": ["watch_date"],
    "ratings": ["rating_date"],
    "reviews": ["review_date"],
    "watchlist": ["added_date"],
}

NUMERIC_COLUMNS = {
    "subscribers": ["age", "monthly_price_usd", "household_size", "tenure_months"],
    "titles": [
        "release_year", "seasons", "content_duration_min", "quality_score",
        "popularity_score", "license_cost_usd", "total_watch_hours", "total_plays"
    ],
    "watch_history": ["content_duration_min", "watch_duration_min", "completion_pct"],
    "ratings": ["rating"],
    "reviews": ["helpful_votes"],
    "watchlist": [],
}

expected_type_rows = []

for table, cols in DATE_COLUMNS.items():
    for col in cols:
        expected_type_rows.append({"table": table, "column": col, "expected_analytical_type": "datetime"})

for table, cols in NUMERIC_COLUMNS.items():
    for col in cols:
        expected_type_rows.append({"table": table, "column": col, "expected_analytical_type": "numeric"})

expected_types = pd.DataFrame(expected_type_rows)
display(expected_types)

,table,column,expected_analytical_type
0,subscribers,signup_date,datetime
1,subscribers,churn_date,datetime
2,titles,date_added,datetime
3,titles,license_expiry,datetime
4,watch_history,watch_date,datetime
5,ratings,rating_date,datetime
6,reviews,review_date,datetime
7,watchlist,added_date,datetime
8,subscribers,age,numeric
9,subscribers,monthly_price_usd,numeric


### 7.2 Date Parsing Validation

In [11]:
# ------------------------------------------------------------
# DATE PARSING VALIDATION
# Convert each expected date field with errors='coerce'.
# If a non-null source value becomes NaT, it means parsing failed.
# ------------------------------------------------------------

date_validation_rows = []

for table, columns in DATE_COLUMNS.items():
    for column in columns:
        original = dfs[table][column]
        parsed = pd.to_datetime(original, errors="coerce")
        parse_failures = int((original.notna() & parsed.isna()).sum())
        date_validation_rows.append({
            "table": table,
            "column": column,
            "non_null_source_values": int(original.notna().sum()),
            "parse_failures": parse_failures,
            "valid_date_format": parse_failures == 0,
            "min_date": parsed.min(),
            "max_date": parsed.max(),
        })

date_validation = pd.DataFrame(date_validation_rows)
display(date_validation)

date_validation.to_csv(OUTPUT_DIR / "date_validation.csv", index=False)

,table,column,non_null_source_values,parse_failures,valid_date_format,min_date,max_date
0,subscribers,signup_date,15000,0,True,2016-01-13,2026-05-31
1,subscribers,churn_date,3801,0,True,2016-04-03,2026-06-30
2,titles,date_added,9000,0,True,2016-01-03,2026-05-31
3,titles,license_expiry,7034,0,True,2025-11-13,2030-07-09
4,watch_history,watch_date,650000,0,True,2017-03-10,2026-05-31
5,ratings,rating_date,130000,0,True,2016-01-13,2026-05-31
6,reviews,review_date,110000,0,True,2016-01-14,2026-05-31
7,watchlist,added_date,65000,0,True,2016-03-03,2026-05-31


### 7.3 Numeric Validation

In [12]:
# ------------------------------------------------------------
# NUMERIC VALIDATION
# Try converting each expected numeric field to a number.
# Any non-null value that becomes NaN is counted as a conversion failure.
# ------------------------------------------------------------

numeric_validation_rows = []

for table, columns in NUMERIC_COLUMNS.items():
    for column in columns:
        source = dfs[table][column]
        coerced = pd.to_numeric(source, errors="coerce")
        conversion_failures = int((source.notna() & coerced.isna()).sum())
        numeric_validation_rows.append({
            "table": table,
            "column": column,
            "source_dtype": str(source.dtype),
            "conversion_failures": conversion_failures,
            "numeric_valid": conversion_failures == 0,
        })

numeric_validation = pd.DataFrame(numeric_validation_rows)
display(numeric_validation)

numeric_validation.to_csv(OUTPUT_DIR / "numeric_validation.csv", index=False)

,table,column,source_dtype,conversion_failures,numeric_valid
0,subscribers,age,int64,0,True
1,subscribers,monthly_price_usd,float64,0,True
2,subscribers,household_size,int64,0,True
3,subscribers,tenure_months,int64,0,True
4,titles,release_year,int64,0,True
5,titles,seasons,int64,0,True
6,titles,content_duration_min,int64,0,True
7,titles,quality_score,float64,0,True
8,titles,popularity_score,float64,0,True
9,titles,license_cost_usd,int64,0,True


### 7.4 Data Type Validation Summary

In [13]:
# Summarize datatype validation into two simple indicators:
# 1) how many date columns contain parsing failures
# 2) how many numeric columns contain conversion failures

datatype_validation_summary = pd.DataFrame({
    "check": [
        "Date columns with parse failures",
        "Numeric columns with conversion failures",
    ],
    "issues_found": [
        int((date_validation["parse_failures"] > 0).sum()),
        int((numeric_validation["conversion_failures"] > 0).sum()),
    ]
})

display(datatype_validation_summary)

,check,issues_found
0,Date columns with parse failures,0
1,Numeric columns with conversion failures,0


## 8. Business & Data Quality Checks

### 8.1 Subscriber Churn-Date Rules

In [14]:
# ------------------------------------------------------------
# SUBSCRIBER CHURN-DATE BUSINESS RULES
# We verify three things:
# - inactive subscribers have a churn date
# - churn happens after signup
# - active subscribers do NOT have a churn date
# ------------------------------------------------------------

subscribers_check = dfs["subscribers"].copy()
subscribers_check["signup_date_dt"] = pd.to_datetime(subscribers_check["signup_date"], errors="coerce")
subscribers_check["churn_date_dt"] = pd.to_datetime(subscribers_check["churn_date"], errors="coerce")

inactive = subscribers_check[~subscribers_check["is_active"]]
active = subscribers_check[subscribers_check["is_active"]]

churn_before_or_on_signup = inactive[
    inactive["churn_date_dt"].notna()
    & (inactive["churn_date_dt"] <= inactive["signup_date_dt"])
]

inactive_missing_churn = inactive[inactive["churn_date_dt"].isna()]
active_with_churn_date = active[active["churn_date_dt"].notna()]

subscriber_churn_checks = pd.DataFrame([
    {
        "check": "Inactive subscribers with missing churn_date",
        "rows_flagged": len(inactive_missing_churn),
    },
    {
        "check": "Churned subscribers with churn_date <= signup_date",
        "rows_flagged": len(churn_before_or_on_signup),
    },
    {
        "check": "Active subscribers with nonblank churn_date",
        "rows_flagged": len(active_with_churn_date),
    },
])

display(subscriber_churn_checks)

,check,rows_flagged
0,Inactive subscribers with missing churn_date,0
1,Churned subscribers with churn_date <= signup_...,0
2,Active subscribers with nonblank churn_date,0


### 8.2 Short-Tenure Active Subscribers

In [15]:
# ------------------------------------------------------------
# SHORT-TENURE ACTIVE SUBSCRIBERS
# The PRD says these records are valid and should only be noted.
# Here, "short tenure" is defined as less than 3 months.
# ------------------------------------------------------------

SHORT_TENURE_MONTHS = 3

short_tenure_active = subscribers_check[
    subscribers_check["is_active"]
    & (subscribers_check["tenure_months"] < SHORT_TENURE_MONTHS)
].copy()

print(f"Active subscribers with tenure < {SHORT_TENURE_MONTHS} months: {len(short_tenure_active):,}")
print(f"Minimum active tenure: {active['tenure_months'].min()} month(s)")

display(short_tenure_active[
    ["subscriber_id", "signup_date", "tenure_months", "plan_type", "country", "is_active"]
].head(10))

Active subscribers with tenure < 3 months: 486
Minimum active tenure: 1 month(s)


,subscriber_id,signup_date,tenure_months,plan_type,country,is_active
66,SUB100066,2026-05-18,1,Basic with Ads,Japan,True
87,SUB100087,2026-03-29,2,Standard,South Korea,True
102,SUB100102,2026-04-30,1,Standard,India,True
144,SUB100144,2026-04-10,1,Premium,South Korea,True
170,SUB100170,2026-04-14,1,Premium,India,True
270,SUB100270,2026-05-05,1,Premium,United States,True
277,SUB100277,2026-04-05,1,Basic with Ads,South Korea,True
279,SUB100279,2026-05-31,1,Standard,Canada,True
326,SUB100326,2026-04-30,1,Premium,Germany,True
358,SUB100358,2026-04-24,1,Premium,South Korea,True


**Interpretation:** The PRD explicitly states that very short-tenure subscribers can legitimately be active. They are therefore documented but not treated as errors.

### 8.3 Watch-Duration Outliers

In [16]:
# ------------------------------------------------------------
# WATCH-DURATION OUTLIERS
# A viewing session should not have more watched minutes
# than the full duration of the content itself.
# We also check for impossible negative watch durations.
# ------------------------------------------------------------

watch_history_check = dfs["watch_history"].copy()

duration_outliers = watch_history_check[
    watch_history_check["watch_duration_min"] > watch_history_check["content_duration_min"]
].copy()

negative_watch_duration = watch_history_check[
    watch_history_check["watch_duration_min"] < 0
].copy()

duration_check_summary = pd.DataFrame([
    {
        "check": "watch_duration_min > content_duration_min",
        "rows_flagged": len(duration_outliers),
    },
    {
        "check": "watch_duration_min < 0",
        "rows_flagged": len(negative_watch_duration),
    }
])

display(duration_check_summary)

,check,rows_flagged
0,watch_duration_min > content_duration_min,0
1,watch_duration_min < 0,0


### 8.4 Completion Percentage Validation

In [17]:
# ------------------------------------------------------------
# COMPLETION PERCENTAGE VALIDATION
# Expected formula:
# completion_pct = watch_duration_min / content_duration_min * 100
# A small difference can occur because stored values are rounded.
# ------------------------------------------------------------

# Recalculate the percentage from raw duration values for comparison.
watch_history_check["expected_completion_pct"] = (
    watch_history_check["watch_duration_min"]
    / watch_history_check["content_duration_min"]
    * 100
)

watch_history_check["completion_diff_pp"] = (
    watch_history_check["completion_pct"]
    - watch_history_check["expected_completion_pct"]
).abs()

# Treat differences up to 1 percentage point as acceptable rounding variation.
COMPLETION_TOLERANCE_PP = 1.0

completion_mismatches = watch_history_check[
    watch_history_check["completion_diff_pp"] > COMPLETION_TOLERANCE_PP
].copy()

completion_range_issues = watch_history_check[
    (watch_history_check["completion_pct"] < 0)
    | (watch_history_check["completion_pct"] > 100)
].copy()

completion_check_summary = pd.DataFrame([
    {
        "check": f"Completion formula mismatch > {COMPLETION_TOLERANCE_PP} percentage point",
        "rows_flagged": len(completion_mismatches),
    },
    {
        "check": "completion_pct outside 0-100",
        "rows_flagged": len(completion_range_issues),
    },
    {
        "check": "Maximum absolute formula difference (percentage points)",
        "rows_flagged": round(float(watch_history_check["completion_diff_pp"].max()), 4),
    },
    {
        "check": "Rows differing by > 0.1 percentage point (rounding diagnostic)",
        "rows_flagged": int((watch_history_check["completion_diff_pp"] > 0.1).sum()),
    },
])

display(completion_check_summary)

,check,rows_flagged
0,Completion formula mismatch > 1.0 percentage p...,0.0000
1,completion_pct outside 0-100,0.0000
2,Maximum absolute formula difference (percentag...,0.1556
3,Rows differing by > 0.1 percentage point (roun...,1845.0000


**Interpretation:** A small sub-percentage-point difference can arise from rounding. Records are only treated as formula failures when the difference exceeds the stated 1 percentage-point tolerance.

### 8.5 Review Sentiment Validation

In [18]:
# ------------------------------------------------------------
# REVIEW SENTIMENT VALIDATION
# The Data Dictionary allows only:
# Positive, Neutral, and Negative.
# Any other value is considered invalid.
# ------------------------------------------------------------

VALID_SENTIMENTS = {"Positive", "Neutral", "Negative"}

reviews_check = dfs["reviews"].copy()
invalid_sentiment = reviews_check[~reviews_check["sentiment"].isin(VALID_SENTIMENTS)]

sentiment_counts = (
    reviews_check["sentiment"]
    .value_counts(dropna=False)
    .rename_axis("sentiment")
    .reset_index(name="review_count")
)

display(sentiment_counts)
print(f"Invalid sentiment rows: {len(invalid_sentiment):,}")

,sentiment,review_count
0,Positive,75007
1,Neutral,22759
2,Negative,12234


Invalid sentiment rows: 0


### 8.6 Watch History Referential Integrity

In [19]:
# ------------------------------------------------------------
# WATCH_HISTORY REFERENTIAL INTEGRITY
# Convert valid parent IDs to Python sets for fast membership checking.
# Then find watch sessions pointing to unknown subscribers or titles.
# ------------------------------------------------------------

subscriber_ids = set(dfs["subscribers"]["subscriber_id"])
title_ids = set(dfs["titles"]["title_id"])

watch_history_orphan_subscribers = dfs["watch_history"][
    ~dfs["watch_history"]["subscriber_id"].isin(subscriber_ids)
]

watch_history_orphan_titles = dfs["watch_history"][
    ~dfs["watch_history"]["title_id"].isin(title_ids)
]

watch_history_ri = pd.DataFrame([
    {
        "relationship": "watch_history.subscriber_id -> subscribers.subscriber_id",
        "orphan_rows": len(watch_history_orphan_subscribers),
    },
    {
        "relationship": "watch_history.title_id -> titles.title_id",
        "orphan_rows": len(watch_history_orphan_titles),
    },
])

display(watch_history_ri)

,relationship,orphan_rows
0,watch_history.subscriber_id -> subscribers.sub...,0
1,watch_history.title_id -> titles.title_id,0


### 8.7 Watchlist Referential Integrity

In [20]:
# ------------------------------------------------------------
# WATCHLIST REFERENTIAL INTEGRITY
# Every saved title must belong to a real subscriber
# and reference a real title in the catalogue.
# ------------------------------------------------------------

watchlist_orphan_subscribers = dfs["watchlist"][
    ~dfs["watchlist"]["subscriber_id"].isin(subscriber_ids)
]

watchlist_orphan_titles = dfs["watchlist"][
    ~dfs["watchlist"]["title_id"].isin(title_ids)
]

watchlist_ri = pd.DataFrame([
    {
        "relationship": "watchlist.subscriber_id -> subscribers.subscriber_id",
        "orphan_rows": len(watchlist_orphan_subscribers),
    },
    {
        "relationship": "watchlist.title_id -> titles.title_id",
        "orphan_rows": len(watchlist_orphan_titles),
    },
])

display(watchlist_ri)

,relationship,orphan_rows
0,watchlist.subscriber_id -> subscribers.subscri...,0
1,watchlist.title_id -> titles.title_id,0


### 8.8 Ratings Validation

In [21]:
# ------------------------------------------------------------
# ADDITIONAL RATINGS VALIDATION
# Although not explicitly required for every Phase 1 bullet,
# we also verify the 1-5 rating range and foreign-key relationships.
# ------------------------------------------------------------

ratings_check = dfs["ratings"].copy()

invalid_ratings = ratings_check[
    (ratings_check["rating"] < 1)
    | (ratings_check["rating"] > 5)
]

ratings_orphan_subscribers = ratings_check[
    ~ratings_check["subscriber_id"].isin(subscriber_ids)
]

ratings_orphan_titles = ratings_check[
    ~ratings_check["title_id"].isin(title_ids)
]

ratings_summary = pd.DataFrame([
    {"check": "Ratings outside 1-5", "rows_flagged": len(invalid_ratings)},
    {"check": "Ratings with unknown subscriber_id", "rows_flagged": len(ratings_orphan_subscribers)},
    {"check": "Ratings with unknown title_id", "rows_flagged": len(ratings_orphan_titles)},
])

display(ratings_summary)

,check,rows_flagged
0,Ratings outside 1-5,0
1,Ratings with unknown subscriber_id,0
2,Ratings with unknown title_id,0


### 8.9 Catalogue Consistency Checks

In [22]:
# ------------------------------------------------------------
# CATALOGUE CONSISTENCY CHECKS
# Validate logical rules in titles.csv, including:
# licence-expiry logic, movie/TV season rules, and score ranges.
# ------------------------------------------------------------

titles_check = dfs["titles"].copy()

original_expiry_mismatch = titles_check[
    titles_check["is_original"]
    & titles_check["license_expiry"].notna()
]

licensed_missing_expiry = titles_check[
    (~titles_check["is_original"])
    & titles_check["license_expiry"].isna()
]

movie_season_issues = titles_check[
    (titles_check["type"] == "Movie")
    & (titles_check["seasons"] != 0)
]

tv_season_issues = titles_check[
    (titles_check["type"] == "TV Show")
    & (titles_check["seasons"] <= 0)
]

quality_score_issues = titles_check[
    (titles_check["quality_score"] < 0)
    | (titles_check["quality_score"] > 100)
]

popularity_range_issues = titles_check[
    (titles_check["popularity_score"] < 0)
    | (titles_check["popularity_score"] > 1)
]

catalogue_checks = pd.DataFrame([
    {"check": "Original titles with nonblank license_expiry", "rows_flagged": len(original_expiry_mismatch)},
    {"check": "Licensed titles with blank license_expiry", "rows_flagged": len(licensed_missing_expiry)},
    {"check": "Movies with seasons != 0", "rows_flagged": len(movie_season_issues)},
    {"check": "TV Shows with seasons <= 0", "rows_flagged": len(tv_season_issues)},
    {"check": "quality_score outside 0-100", "rows_flagged": len(quality_score_issues)},
    {"check": "popularity_score outside documented 0-1 range", "rows_flagged": len(popularity_range_issues)},
])

display(catalogue_checks)

,check,rows_flagged
0,Original titles with nonblank license_expiry,0
1,Licensed titles with blank license_expiry,0
2,Movies with seasons != 0,0
3,TV Shows with seasons <= 0,0
4,quality_score outside 0-100,0
5,popularity_score outside documented 0-1 range,1285


**Analyst note:** `popularity_score` contains values above the documented 0–1 range. Because the correct intended scale cannot be inferred reliably from the supplied files, these values are **flagged but not altered**.

### 8.10 Additional Temporal Consistency Checks

In [23]:
# ------------------------------------------------------------
# ADDITIONAL TEMPORAL CONSISTENCY CHECKS
# These checks go beyond the minimum PRD requirements.
# We compare event dates with subscriber signup/churn dates
# and the date a title was added to the catalogue.
# ------------------------------------------------------------

subscriber_dates = dfs["subscribers"][
    ["subscriber_id", "signup_date", "churn_date", "is_active"]
].copy()
subscriber_dates["signup_date"] = pd.to_datetime(subscriber_dates["signup_date"], errors="coerce")
subscriber_dates["churn_date"] = pd.to_datetime(subscriber_dates["churn_date"], errors="coerce")

title_dates = dfs["titles"][["title_id", "date_added"]].copy()
title_dates["date_added"] = pd.to_datetime(title_dates["date_added"], errors="coerce")

event_date_columns = {
    "watch_history": "watch_date",
    "ratings": "rating_date",
    "reviews": "review_date",
    "watchlist": "added_date",
}

temporal_rows = []

# Apply the same lifecycle-date checks to each event table.
for table, event_date_col in event_date_columns.items():
    # Join subscriber lifecycle dates and title-added dates onto each event.
    event_df = (
        dfs[table]
        .merge(subscriber_dates, on="subscriber_id", how="left")
        .merge(title_dates, on="title_id", how="left")
    )
    event_df[event_date_col] = pd.to_datetime(event_df[event_date_col], errors="coerce")

    temporal_rows.append({
        "table": table,
        "event_before_subscriber_signup": int(
            (event_df[event_date_col] < event_df["signup_date"]).sum()
        ),
        "event_before_title_added": int(
            (event_df[event_date_col] < event_df["date_added"]).sum()
        ),
        "event_after_recorded_churn": int(
            ((~event_df["is_active"]) & (event_df[event_date_col] > event_df["churn_date"])).sum()
        ),
    })

temporal_checks = pd.DataFrame(temporal_rows)
display(temporal_checks)

temporal_checks.to_csv(OUTPUT_DIR / "additional_temporal_checks.csv", index=False)

,table,event_before_subscriber_signup,event_before_title_added,event_after_recorded_churn
0,watch_history,0,0,131199
1,ratings,25786,0,21359
2,reviews,22102,0,17739
3,watchlist,0,30219,7781


**Interpretation:** These lifecycle checks go beyond the explicit Phase 1 checklist. They reveal temporal inconsistencies in some event tables. Because the PRD does not provide a correction rule for them, the records are retained and the issue is documented for later interpretation rather than deleted or rewritten.

## 9. Data Cleaning

In [24]:
# ------------------------------------------------------------
# DATA CLEANING
# Create deep copies so the original raw DataFrames remain untouched.
# We standardize confirmed date fields but do NOT invent corrections
# for ambiguous issues that lack a reliable business rule.
# ------------------------------------------------------------

# Deep copies ensure our raw source DataFrames stay unchanged.
clean_dfs = {
    table: df.copy(deep=True)
    for table, df in dfs.items()
}

# Convert documented date fields from strings into true datetime values.
for table, columns in DATE_COLUMNS.items():
    for column in columns:
        clean_dfs[table][column] = pd.to_datetime(
            clean_dfs[table][column],
            errors="coerce"
        )

print("Cleaning actions completed.")
print("Rows removed: 0")
print("Ambiguous values corrected: 0")
print("Documented date columns converted to datetime.")

Cleaning actions completed.
Rows removed: 0
Ambiguous values corrected: 0
Documented date columns converted to datetime.


### Cleaning Decisions

- No duplicate records are removed because none were found.
- No missing value is imputed where the Data Dictionary defines the blank as meaningful.
- No watch-duration records are removed because none exceed content duration.
- No sentiment values require correction.
- `popularity_score` is not rescaled because the intended scale is ambiguous.
- Additional temporal inconsistencies are retained and documented because no reliable correction rule is provided.

## 10. Post-Cleaning Validation

In [25]:
# ------------------------------------------------------------
# POST-CLEANING VALIDATION
# Confirm that cleaning did not accidentally change row counts
# or introduce duplicate rows/keys.
# We also verify that date columns are now true datetime columns.
# ------------------------------------------------------------

post_validation_rows = []

for table, df in clean_dfs.items():
    post_validation_rows.append({
        "table": table,
        "rows_before": len(dfs[table]),
        "rows_after": len(df),
        "row_count_preserved": len(dfs[table]) == len(df),
        "exact_duplicates_after": int(df.duplicated().sum()),
        "duplicate_pk_after": int(df.duplicated(subset=[PRIMARY_KEYS[table]]).sum()),
    })

post_cleaning_validation = pd.DataFrame(post_validation_rows)
display(post_cleaning_validation)

date_dtype_check = []
for table, columns in DATE_COLUMNS.items():
    for column in columns:
        date_dtype_check.append({
            "table": table,
            "column": column,
            "dtype_after_cleaning": str(clean_dfs[table][column].dtype),
            "is_datetime": pd.api.types.is_datetime64_any_dtype(clean_dfs[table][column]),
        })

display(pd.DataFrame(date_dtype_check))

,table,rows_before,rows_after,row_count_preserved,exact_duplicates_after,duplicate_pk_after
0,subscribers,15000,15000,True,0,0
1,titles,9000,9000,True,0,0
2,watch_history,650000,650000,True,0,0
3,ratings,130000,130000,True,0,0
4,reviews,110000,110000,True,0,0
5,watchlist,65000,65000,True,0,0


,table,column,dtype_after_cleaning,is_datetime
0,subscribers,signup_date,datetime64[us],True
1,subscribers,churn_date,datetime64[us],True
2,titles,date_added,datetime64[us],True
3,titles,license_expiry,datetime64[us],True
4,watch_history,watch_date,datetime64[us],True
5,ratings,rating_date,datetime64[us],True
6,reviews,review_date,datetime64[us],True
7,watchlist,added_date,datetime64[us],True


## 11. Export Cleaned Data

In [26]:
# ------------------------------------------------------------
# EXPORT CLEANED DATA
# Save one cleaned CSV per source table.
# These files will become the input for Phase 2 EDA and later analysis.
# ------------------------------------------------------------

exported_files = []

# Write each cleaned table into the dedicated cleaned-data folder.
for table, df in clean_dfs.items():
    output_path = CLEAN_DIR / f"{table}_clean.csv"
    df.to_csv(output_path, index=False, date_format="%Y-%m-%d")
    exported_files.append({
        "table": table,
        "rows": len(df),
        "output_file": str(output_path),
    })

export_summary = pd.DataFrame(exported_files)
display(export_summary)

print(f"Exported {len(export_summary)} cleaned CSV files.")

,table,rows,output_file
0,subscribers,15000,..\data\cleaned\subscribers_clean.csv
1,titles,9000,..\data\cleaned\titles_clean.csv
2,watch_history,650000,..\data\cleaned\watch_history_clean.csv
3,ratings,130000,..\data\cleaned\ratings_clean.csv
4,reviews,110000,..\data\cleaned\reviews_clean.csv
5,watchlist,65000,..\data\cleaned\watchlist_clean.csv


Exported 6 cleaned CSV files.


## 12. Data Quality Summary

In [27]:
# ------------------------------------------------------------
# FINAL DATA QUALITY SUMMARY
# Bring the most important validation results into one compact table
# so the overall health of the dataset is easy to review.
# ------------------------------------------------------------

all_event_ri = pd.DataFrame([
    {
        "table": table,
        "unknown_subscriber_ids": int((~dfs[table]["subscriber_id"].isin(subscriber_ids)).sum()),
        "unknown_title_ids": int((~dfs[table]["title_id"].isin(title_ids)).sum()),
    }
    for table in ["watch_history", "ratings", "reviews", "watchlist"]
])

core_summary = pd.DataFrame([
    {"metric": "Raw datasets", "value": len(dfs)},
    {"metric": "Total raw records", "value": total_rows},
    {"metric": "Exact duplicate rows across all tables", "value": int(sum(df.duplicated().sum() for df in dfs.values()))},
    {"metric": "Duplicate primary-key rows across all tables", "value": int(sum(df.duplicated(subset=[PRIMARY_KEYS[t]]).sum() for t, df in dfs.items()))},
    {"metric": "Date parse failures", "value": int(date_validation["parse_failures"].sum())},
    {"metric": "Numeric conversion failures", "value": int(numeric_validation["conversion_failures"].sum())},
    {"metric": "Watch-duration sessions longer than content", "value": len(duration_outliers)},
    {"metric": "Completion formula failures >1pp", "value": len(completion_mismatches)},
    {"metric": "Invalid review sentiments", "value": len(invalid_sentiment)},
    {"metric": "Watch-history orphan FK rows", "value": len(watch_history_orphan_subscribers) + len(watch_history_orphan_titles)},
    {"metric": "Watchlist orphan FK rows", "value": len(watchlist_orphan_subscribers) + len(watchlist_orphan_titles)},
    {"metric": "Active subscribers with churn_date", "value": len(active_with_churn_date)},
    {"metric": "Churned subscribers with churn_date <= signup_date", "value": len(churn_before_or_on_signup)},
    {"metric": "Short-tenure active subscribers (<3 months; informational)", "value": len(short_tenure_active)},
    {"metric": "Popularity-score rows outside documented 0-1 range", "value": len(popularity_range_issues)},
])

display(core_summary)

core_summary.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)
all_event_ri.to_csv(OUTPUT_DIR / "referential_integrity.csv", index=False)

,metric,value
0,Raw datasets,6
1,Total raw records,979000
2,Exact duplicate rows across all tables,0
3,Duplicate primary-key rows across all tables,0
4,Date parse failures,0
5,Numeric conversion failures,0
6,Watch-duration sessions longer than content,0
7,Completion formula failures >1pp,0
8,Invalid review sentiments,0
9,Watch-history orphan FK rows,0


In [28]:
# ------------------------------------------------------------
# DATA QUALITY ISSUE LOG
# Separate expected conditions from genuine warnings.
# This makes it clear what was preserved intentionally
# and what analysts should keep in mind in later phases.
# ------------------------------------------------------------

issue_log = pd.DataFrame([
    {
        "issue": "Active subscriber churn_date blanks",
        "severity": "Expected / informational",
        "rows": int(dfs["subscribers"]["churn_date"].isna().sum()),
        "action": "Preserved; expected for active subscribers",
    },
    {
        "issue": "Original-title license_expiry blanks",
        "severity": "Expected / informational",
        "rows": int(dfs["titles"]["license_expiry"].isna().sum()),
        "action": "Preserved; expected for Originals",
    },
    {
        "issue": "popularity_score outside documented 0-1 range",
        "severity": "Warning",
        "rows": len(popularity_range_issues),
        "action": "Flagged; not corrected because intended scale is ambiguous",
    },
    {
        "issue": "Additional temporal inconsistencies across event tables",
        "severity": "Warning",
        "rows": int(
            temporal_checks[
                ["event_before_subscriber_signup", "event_before_title_added", "event_after_recorded_churn"]
            ].sum().sum()
        ),
        "action": "Retained; outside explicit PRD rules and no reliable correction rule supplied",
    },
])

display(issue_log)
issue_log.to_csv(OUTPUT_DIR / "data_quality_issue_log.csv", index=False)

,issue,severity,rows,action
0,Active subscriber churn_date blanks,Expected / informational,11199,Preserved; expected for active subscribers
1,Original-title license_expiry blanks,Expected / informational,1966,Preserved; expected for Originals
2,popularity_score outside documented 0-1 range,Warning,1285,Flagged; not corrected because intended scale ...
3,Additional temporal inconsistencies across eve...,Warning,256185,Retained; outside explicit PRD rules and no re...


## 13. Phase 1 Conclusion

In [29]:
# ------------------------------------------------------------
# PHASE 1 CONCLUSION
# Count core PRD failures and generate a written conclusion
# using the actual calculated results rather than hardcoded numbers.
# ------------------------------------------------------------

core_failures = (
    int(exact_duplicate_summary["exact_duplicate_rows"].sum())
    + int(pk_duplicate_summary["duplicate_pk_rows"].sum())
    + int(date_validation["parse_failures"].sum())
    + int(numeric_validation["conversion_failures"].sum())
    + len(duration_outliers)
    + len(completion_mismatches)
    + len(invalid_sentiment)
    + len(watch_history_orphan_subscribers)
    + len(watch_history_orphan_titles)
    + len(watchlist_orphan_subscribers)
    + len(watchlist_orphan_titles)
    + len(active_with_churn_date)
    + len(churn_before_or_on_signup)
)

conclusion = f"""
### Phase 1 Conclusion

The six StreamFlix datasets contain **{total_rows:,} raw records**. All required Phase 1 checks were completed.

The core PRD validation found **{core_failures:,} failures** across duplicate keys/rows, date parsing, numeric conversion, invalid watch-duration logic, completion-formula tolerance, review sentiment, required referential-integrity checks, and subscriber churn-date consistency.

Expected nulls were preserved rather than imputed. **{len(short_tenure_active):,} active subscribers have tenure below three months; this is documented as valid according to the project brief.**

Two additional analyst observations require documentation rather than automatic correction:

1. **{len(popularity_range_issues):,} titles** have `popularity_score` outside the 0-1 range described in the Data Dictionary.
2. Additional lifecycle/date consistency checks identify event-date inconsistencies in some event tables.

Because no reliable correction rule is supplied for these two observations, the source values are retained unchanged. Cleaned copies standardize date fields while preserving valid records.
"""

display(Markdown(conclusion))


### Phase 1 Conclusion

The six StreamFlix datasets contain **979,000 raw records**. All required Phase 1 checks were completed.

The core PRD validation found **0 failures** across duplicate keys/rows, date parsing, numeric conversion, invalid watch-duration logic, completion-formula tolerance, review sentiment, required referential-integrity checks, and subscriber churn-date consistency.

Expected nulls were preserved rather than imputed. **486 active subscribers have tenure below three months; this is documented as valid according to the project brief.**

Two additional analyst observations require documentation rather than automatic correction:

1. **1,285 titles** have `popularity_score` outside the 0-1 range described in the Data Dictionary.
2. Additional lifecycle/date consistency checks identify event-date inconsistencies in some event tables.

Because no reliable correction rule is supplied for these two observations, the source values are retained unchanged. Cleaned copies standardize date fields while preserving valid records.


## 14. Phase 1 Data Quality Report

In [30]:
# ------------------------------------------------------------
# ONE-PAGE DATA QUALITY REPORT
# Build a management-friendly report directly from our calculated results.
# This keeps every number traceable back to the notebook analysis.
# ------------------------------------------------------------

temporal_total = int(
    temporal_checks[
        ["event_before_subscriber_signup", "event_before_title_added", "event_after_recorded_churn"]
    ].sum().sum()
)

report_md = f"""
### StreamFlix — Phase 1 Data Quality Report

**Scope.** The Phase 1 audit reviewed **6 interconnected tables containing {total_rows:,} raw records**. The assessment covered table structure, missing values, duplicates, data types, watch-duration validity, completion-percentage consistency, subscriber churn logic, review sentiment, and foreign-key integrity. Raw CSV files were treated as read-only and all standardization was applied to copies.

**Completeness.** Missing values occur only in fields whose blanks have a documented business meaning. `churn_date` is blank for active subscribers, while `license_expiry` is blank for StreamFlix Originals. These values were preserved rather than filled.

**Uniqueness and integrity.** No exact duplicate rows or duplicate primary keys were found. All `subscriber_id` and `title_id` values in `watch_history` and `watchlist` link to valid parent records. Additional checks also found no orphan keys in ratings or reviews.

**Type and rule validation.** All documented dates parse successfully and all expected numeric fields convert successfully. No viewing session has `watch_duration_min` greater than `content_duration_min`. No completion record differs from the duration-based formula by more than the chosen **1 percentage-point tolerance**. Review sentiment contains only Positive, Neutral, or Negative. Churned subscribers have valid churn dates after signup, and active subscribers have blank churn dates.

**Informational observations.** **{len(short_tenure_active):,} active subscribers** have tenure below three months; the PRD states this is acceptable. The Data Dictionary describes `popularity_score` as 0-1, but **{len(popularity_range_issues):,} title records** exceed 1. Because the intended scale cannot be established reliably, these values were flagged and left unchanged. Additional lifecycle checks found **{temporal_total:,} temporal flags** across event timing rules; these were retained because the PRD does not provide a correction rule.

**Cleaning outcome.** No valid records were deleted. Date fields were standardized to datetime in the cleaned working copies, expected nulls were preserved, and ambiguous quality issues were documented rather than silently modified. The resulting cleaned datasets are suitable for Phase 2 EDA, provided the documented warning fields are interpreted carefully.
"""

display(Markdown(report_md))

# Save a reusable text copy of the report alongside the QC outputs.
with open(OUTPUT_DIR / "Phase1_DataQuality_Report.md", "w", encoding="utf-8") as f:
    f.write(report_md)


### StreamFlix — Phase 1 Data Quality Report

**Scope.** The Phase 1 audit reviewed **6 interconnected tables containing 979,000 raw records**. The assessment covered table structure, missing values, duplicates, data types, watch-duration validity, completion-percentage consistency, subscriber churn logic, review sentiment, and foreign-key integrity. Raw CSV files were treated as read-only and all standardization was applied to copies.

**Completeness.** Missing values occur only in fields whose blanks have a documented business meaning. `churn_date` is blank for active subscribers, while `license_expiry` is blank for StreamFlix Originals. These values were preserved rather than filled.

**Uniqueness and integrity.** No exact duplicate rows or duplicate primary keys were found. All `subscriber_id` and `title_id` values in `watch_history` and `watchlist` link to valid parent records. Additional checks also found no orphan keys in ratings or reviews.

**Type and rule validation.** All documented dates parse successfully and all expected numeric fields convert successfully. No viewing session has `watch_duration_min` greater than `content_duration_min`. No completion record differs from the duration-based formula by more than the chosen **1 percentage-point tolerance**. Review sentiment contains only Positive, Neutral, or Negative. Churned subscribers have valid churn dates after signup, and active subscribers have blank churn dates.

**Informational observations.** **486 active subscribers** have tenure below three months; the PRD states this is acceptable. The Data Dictionary describes `popularity_score` as 0-1, but **1,285 title records** exceed 1. Because the intended scale cannot be established reliably, these values were flagged and left unchanged. Additional lifecycle checks found **256,185 temporal flags** across event timing rules; these were retained because the PRD does not provide a correction rule.

**Cleaning outcome.** No valid records were deleted. Date fields were standardized to datetime in the cleaned working copies, expected nulls were preserved, and ambiguous quality issues were documented rather than silently modified. The resulting cleaned datasets are suitable for Phase 2 EDA, provided the documented warning fields are interpreted carefully.


## 15. PRD Requirement Checklist

In [31]:
# ------------------------------------------------------------
# PRD REQUIREMENT CHECKLIST
# Confirm that every requested Phase 1 requirement has been completed.
# This is useful before final submission.
# ------------------------------------------------------------

prd_checklist = pd.DataFrame([
    {"requirement": "Load all 6 CSV files", "status": "Complete"},
    {"requirement": "Profile row count, column count, and data types", "status": "Complete"},
    {"requirement": "Check nulls and null percentages in every column", "status": "Complete"},
    {"requirement": "Check duplicate rows and key duplicates", "status": "Complete"},
    {"requirement": "Validate dates and numeric data types", "status": "Complete"},
    {"requirement": "Flag watch_duration_min > content_duration_min", "status": "Complete"},
    {"requirement": "Verify watch_history subscriber/title referential integrity", "status": "Complete"},
    {"requirement": "Validate churn_date > signup_date for churned subscribers", "status": "Complete"},
    {"requirement": "Validate active subscribers have blank churn_date", "status": "Complete"},
    {"requirement": "Validate completion_pct against duration formula", "status": "Complete"},
    {"requirement": "Validate review sentiment values", "status": "Complete"},
    {"requirement": "Validate watchlist links to real subscribers and titles", "status": "Complete"},
    {"requirement": "Note short-tenure active subscribers", "status": "Complete"},
    {"requirement": "Write 1-page Data Quality Report", "status": "Complete"},
    {"requirement": "Export cleaned working copies", "status": "Complete"},
])

display(prd_checklist)
prd_checklist.to_csv(OUTPUT_DIR / "prd_requirement_checklist.csv", index=False)

,requirement,status
0,Load all 6 CSV files,Complete
1,"Profile row count, column count, and data types",Complete
2,Check nulls and null percentages in every column,Complete
3,Check duplicate rows and key duplicates,Complete
4,Validate dates and numeric data types,Complete
5,Flag watch_duration_min > content_duration_min,Complete
6,Verify watch_history subscriber/title referent...,Complete
7,Validate churn_date > signup_date for churned ...,Complete
8,Validate active subscribers have blank churn_date,Complete
9,Validate completion_pct against duration formula,Complete


## 16. Phase 1 Completion Status

In [32]:
# ------------------------------------------------------------
# PHASE COMPLETION STATUS
# Final console summary showing that Phase 1 has finished successfully
# and the project is ready to move into Phase 2.
# ------------------------------------------------------------

print("=" * 72)
print("STREAMFLIX PHASE 1 — DATA CLEANING & QUALITY CHECK")
print("=" * 72)
print("Status        : COMPLETE")
print(f"Datasets      : {len(dfs)}")
print(f"Raw records   : {total_rows:,}")
print("Raw files     : PRESERVED / READ-ONLY")
print("Cleaned files : EXPORTED")
print("QC outputs    : EXPORTED")
print("Next phase    : Phase 2 — Exploratory Data Analysis")
print("=" * 72)

STREAMFLIX PHASE 1 — DATA CLEANING & QUALITY CHECK
Status        : COMPLETE
Datasets      : 6
Raw records   : 979,000
Raw files     : PRESERVED / READ-ONLY
Cleaned files : EXPORTED
QC outputs    : EXPORTED
Next phase    : Phase 2 — Exploratory Data Analysis
